# Jelle Somali TTS — Kaggle First Pass

هدف هذه الدفترية: تشغيل **Jelle فقط** على GPU، إنتاج عينات صومالية قصيرة، وحفظ النتائج. لا يوجد تدريب أو استنساخ صوت في هذه المرحلة، ولا يتم رفع أي تسجيل شخصي.

**قبل التشغيل:** فعّل Internet وGPU من Notebook → Session options. استخدم GPU متاحًا (T4 أو P100). شغّل الخلايا بالترتيب.

النموذج: `Somali-tts/Jelle_tts` — Revision مثبت لتكون التجربة قابلة للتكرار. المتحدث التجريبي يأتي من مجموعة CMU Arctic العامة، وليس صوت المستخدم.


In [ ]:
# تثبيت إصدارات متوافقة مع بطاقة نموذج Jelle
%pip install -q transformers==4.52.4 datasets==3.6.0 soundfile==0.13.1 accelerate==1.7.0


In [ ]:
import os, json, time, platform
import torch
import soundfile as sf
from IPython.display import Audio, display
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForTextToSpectrogram, SpeechT5HifiGan

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU is not enabled. Turn on GPU in Kaggle Session options, then rerun.')

DEVICE = 'cuda'
OUT_DIR = '/kaggle/working/jelle_first_pass'
os.makedirs(OUT_DIR, exist_ok=True)


## 1. تحميل Jelle و SpeechT5 vocoder

حجم النموذج وحده كبير نسبيًا؛ سيُحمّل إلى مساحة Kaggle المؤقتة. لا تضع هذه الأوزان على Render Free.


In [ ]:
MODEL_ID = 'Somali-tts/Jelle_tts'
MODEL_REVISION = 'ee0109720255e9b979bec7d9787de456747f4fa7'
VOCODER_ID = 'microsoft/speecht5_hifigan'

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForTextToSpectrogram.from_pretrained(MODEL_ID, revision=MODEL_REVISION).to(DEVICE)
vocoder = SpeechT5HifiGan.from_pretrained(VOCODER_ID).to(DEVICE)
model.eval(); vocoder.eval()
print(f'Models loaded in {time.time()-t0:.1f}s')
print('Model class:', type(model).__name__)


## 2. متجه متحدث تجريبي عام

SpeechT5 يحتاج متجه متحدث بطول 512. نستخدم متجهًا عامًا من CMU Arctic لتأكيد مسار التوليد فقط. هذا **ليس** صوت Muuse أو Ubax الصوماليين، ولا استنساخًا لصوتك.


In [ ]:
xvectors = load_dataset('Matthijs/cmu-arctic-xvectors', split='validation')
print('Rows:', len(xvectors))
print('Columns:', xvectors.column_names)
row = xvectors[0]
speaker_embeddings = torch.tensor(row['xvector'], dtype=torch.float32).unsqueeze(0).to(DEVICE)
assert speaker_embeddings.shape == (1, 512), speaker_embeddings.shape
print('Speaker embedding shape:', tuple(speaker_embeddings.shape))
print('Public sample speaker_id:', row.get('speaker_id', 'not provided'))


## 3. توليد ثلاث عينات صومالية قصيرة

ابدأ بالنصوص القصيرة لتقييم النطق قبل تجربة النصوص الطويلة. لا نقسم النص إلى مقاطع في هذه المرحلة حتى لا نخلط جودة النموذج بمشكلات تقطيع النص.


In [ ]:
TEST_TEXTS = [
    'Maanta waxaan tijaabinaynaa codka af Soomaaliga.',
    'Soomaaliya waa dal leh taariikh dheer iyo dhaqan hodan ah.',
    'Fadlan si tartiib ah u akhri qoraalkan, adigoo si cad u dhawaaqaya eray kasta.'
]

def synthesize_jelle(text):
    inputs = processor(text=text, return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        speech = model.generate_speech(
            inputs['input_ids'],
            speaker_embeddings,
            vocoder=vocoder
        )
    audio = speech.detach().float().cpu().numpy()
    if audio.ndim > 1:
        audio = audio.squeeze()
    return audio

results = []
for i, text in enumerate(TEST_TEXTS, start=1):
    started = time.time()
    audio = synthesize_jelle(text)
    path = os.path.join(OUT_DIR, f'jelle_sample_{i}.wav')
    sf.write(path, audio, 16000)
    duration = len(audio) / 16000
    elapsed = time.time() - started
    results.append({'sample': i, 'text': text, 'file': path, 'duration_seconds': round(duration, 2), 'generation_seconds': round(elapsed, 2)})
    print(f'\nSample {i}: {duration:.2f}s audio | {elapsed:.2f}s generation | {path}')
    display(Audio(audio, rate=16000))


## 4. حفظ تقرير التجربة

راجع الصوت بنفسك. سجّل هل الكلمات مفهومة، وهل توجد تكرارات أو تقطعات أو توقفات غريبة. لا تعتمد على رقم Loss وحده للحكم على جودة النطق.


In [ ]:
report = {
    'model': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'vocoder': VOCODER_ID,
    'device': torch.cuda.get_device_name(0),
    'torch_version': torch.__version__,
    'transformers_version': __import__('transformers').__version__,
    'speaker_embedding_source': 'Matthijs/cmu-arctic-xvectors validation[0] (public test voice; not user voice)',
    'samples': results,
    'voice_cloning': False,
    'notes': 'Listen to every sample; record intelligibility, Somali pronunciation, glitches, repetitions and pauses.'
}
report_path = os.path.join(OUT_DIR, 'jelle_test_report.json')
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(json.dumps(report, ensure_ascii=False, indent=2))

print('\nFiles saved in:', OUT_DIR)
print('Download the folder from Kaggle Output after saving a notebook version.')


## معيار النجاح للمرحلة الأولى

لا نربط Jelle بواجهة المستخدم قبل التأكد من:
1. تشغيل العينات الثلاث دون أخطاء.
2. وضوح الكلمات الصومالية عند الاستماع.
3. عدم وجود تكرار/تقطيع شديد أو صمت غير طبيعي.
4. تسجيل زمن التوليد وحجم الملفات.

بعد نجاح ذلك ننتقل لاختبار نص أطول، ثم نبحث منفصلًا في أصوات ذكورية/أنثوية مناسبة. متجه CMU الحالي مؤقت ولا يثبت دعم Muuse وUbax.
